<a href="https://colab.research.google.com/github/yadhurajsp-cell/vision-Transformer-from-scratch/blob/main/vision_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import torch
import torchvision
import matplotlib.pyplot as plt
import torch.utils.data as dataloader
import torch.nn as nn


In [34]:
#getting PIL data
train_datatset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=torchvision.transforms.ToTensor())
test_datatset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=torchvision.transforms.ToTensor())

In [35]:
#define variables
batch_size = 64
learning_rate = 0.001
num_epochs = 10
patch_size = 7
img_size = 28
num_channels = 1
patch_num = (img_size // patch_size) ** 2
num_classes = 10
attention_heads = 4
embed_dim = 20
transformer_blocks = 4
mlp_nodes = 64

In [36]:
train_data = dataloader.DataLoader(train_datatset, batch_size=batch_size, shuffle=True)
test_data = dataloader.DataLoader(test_datatset, batch_size=batch_size, shuffle=True)

In [37]:
#class for patch embedding

class Patch_embedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.embed = nn.Conv2d(num_channels , embed_dim, kernel_size=patch_size, stride=patch_size)

  def forward(self,x):
    x = self.embed(x)
    x = x.flatten(2).transpose(1,2)
    return x





In [52]:
#class for transformer encoder
#layer normalisation
#multihead attention
#layer normalisation
#residulas
#MLP - activation funciton

class Transformer_encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embed_dim)
    self.multihead_attention = nn.MultiheadAttention(embed_dim, num_heads=attention_heads, batch_first=True)
    self.layer_norm2 = nn.LayerNorm(embed_dim)
    self.mlp = nn.Sequential(
        nn.Linear(embed_dim, mlp_nodes),
        nn.GELU(),
        nn.Linear(mlp_nodes,embed_dim)
    )

  def forward(self,x):
   residual1 = x
   x = self.layer_norm1(x)
   x = self.multihead_attention(x,x,x)[0] + residual1
   residual2 = x
   x = self.layer_norm2(x)
   x = self.mlp(x) + residual2
   return x

In [44]:
#class for mlp head for classification
class mlp_head(nn.Module):
  def __init__(self):
    super().__init__()
    self.mlphead = nn.Sequential(
        nn.LayerNorm(embed_dim),
        nn.Linear(embed_dim, embed_dim),
        nn.Linear(embed_dim , num_classes)
    )

  def forward(self,x):
    x = x[:,0]
    x = self.mlphead(x)
    return x

In [45]:
class vit(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding = Patch_embedding()
    self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
    self.pos_embedding = nn.Parameter(torch.randn(1, patch_num + 1, embed_dim))
    self.transformer_encoder = Transformer_encoder()
    self.mlp_head = mlp_head()

  def forward(self,x):
    x = self.patch_embedding(x)
    cls_token = self.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    x = x + self.pos_embedding
    x = self.transformer_encoder(x)
    x = self.mlp_head(x)
    return x

In [46]:
#optimiswer
#loss
#device

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = vit().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

In [53]:
# Re-initialize the model to use the updated Transformer_encoder class
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = vit().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

for i in range(num_epochs):

  train_acc = 0
  total_images = 0
  correct_images = 0

  model.train()

  for images, labels in train_data:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

    prediction = outputs.argmax(dim=1)
    total_images += labels.size(0)
    correct_images += (prediction == labels).sum().item()
    train_acc = correct_images / total_images
  print(f"Epoch {i+1} Accuracy: {train_acc:.4f}")

Epoch 1 Accuracy: 0.6951
Epoch 2 Accuracy: 0.8829
Epoch 3 Accuracy: 0.9074
Epoch 4 Accuracy: 0.9217
Epoch 5 Accuracy: 0.9299
Epoch 6 Accuracy: 0.9367
Epoch 7 Accuracy: 0.9415
Epoch 8 Accuracy: 0.9457
Epoch 9 Accuracy: 0.9480
Epoch 10 Accuracy: 0.9499


In [ ]:
images , labes = next(iter(train_data))
print(images.shape)
embed = nn.Conv2d(num_channels , embed_dim, kernel_size=patch_size, stride=patch_size)
print(embed(images).shape)
print((embed(images).flatten(2).transpose(1,2).shape))